In [1]:
import pandas as pd 
import numpy as np 

In [4]:
df = pd.read_csv('users.csv')
df1 = pd.read_csv('cards.csv')
df2 = pd.read_csv('trans.csv')
df3 = pd.read_csv('fraud.csv')
df4 = pd.read_csv('segmentation.csv')
df5 = pd.read_csv('user_recommendation_base.csv')
df6 = pd.read_csv('interaction_dataset_weak.csv')
df7 = pd.read_csv('card_master_table.csv')
df8 = pd.read_csv('pairwise_training_data.csv')

In [18]:
# Force user features to numeric
df5[user_feat_cols] = df5[user_feat_cols].apply(
    pd.to_numeric, errors='raise'
)

In [19]:
# Force card features to numeric
df7[card_feat_cols] = df7[card_feat_cols].apply(
    pd.to_numeric, errors='raise'
)

In [20]:
print(df5[user_feat_cols].dtypes)
print(df7[card_feat_cols].dtypes)

txn_count             int64
total_spend         float64
avg_txn_amount      float64
category_entropy    float64
num_cards             int64
avg_credit_limit    float64
fico_score            int64
avg_fraud_score     float64
pct_high            float64
pct_critical        float64
cluster               int64
dtype: object
annual_fee            int64
apr                 float64
credit_limit_min      int64
credit_limit_max      int64
reward_grocery        int64
reward_dining         int64
reward_fuel           int64
reward_travel         int64
reward_retail         int64
reward_digital        int64
reward_other          int64
cluster_id            int64
dtype: object


In [21]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

In [22]:
user_feat_cols = [
    'txn_count',
    'total_spend',
    'avg_txn_amount',
    'category_entropy',
    'num_cards',
    'avg_credit_limit',
    'fico_score',
    'avg_fraud_score',
    'pct_high',
    'pct_critical',
    'cluster'
]

In [23]:
card_feat_cols = [
    'annual_fee',
    'apr',
    'credit_limit_min',
    'credit_limit_max',
    'reward_grocery',
    'reward_dining',
    'reward_fuel',
    'reward_travel',
    'reward_retail',
    'reward_digital',
    'reward_other',
    'cluster_id'
]

In [24]:
user_id_map = {uid: idx for idx, uid in enumerate(df5['User'].unique())}
card_id_map = {cid: idx for idx, cid in enumerate(df7['card_id'].unique())}

In [25]:
class PairwiseNCFDataset(Dataset):
    def __init__(self, pairwise_df, user_df, card_df):
        self.pairs = pairwise_df.reset_index(drop=True)
        self.user_df = user_df.set_index('User')
        self.card_df = card_df.set_index('card_id')

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        row = self.pairs.iloc[idx]

        user_id = int(row['user_id'])
        pos_card = row['pos_card_id']
        neg_card = row['neg_card_id']

        user_feats = torch.tensor(
            self.user_df.loc[user_id, user_feat_cols].values,
            dtype=torch.float32
        )

        pos_feats = torch.tensor(
            self.card_df.loc[pos_card, card_feat_cols].values,
            dtype=torch.float32
        )

        neg_feats = torch.tensor(
            self.card_df.loc[neg_card, card_feat_cols].values,
            dtype=torch.float32
        )

        return (
            torch.tensor(user_id_map[user_id]),
            user_feats,
            torch.tensor(card_id_map[pos_card]),
            pos_feats,
            torch.tensor(card_id_map[neg_card]),
            neg_feats
        )

In [26]:
class UserEncoder(nn.Module):
    def __init__(self, num_users, user_feat_dim, emb_dim=32):
        super().__init__()
        self.emb = nn.Embedding(num_users, emb_dim)
        self.mlp = nn.Sequential(
            nn.Linear(user_feat_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32)
        )

    def forward(self, uid, feats):
        return torch.cat([self.emb(uid), self.mlp(feats)], dim=1)

In [27]:
class CardEncoder(nn.Module):
    def __init__(self, num_cards, card_feat_dim, emb_dim=32):
        super().__init__()
        self.emb = nn.Embedding(num_cards, emb_dim)
        self.mlp = nn.Sequential(
            nn.Linear(card_feat_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32)
        )

    def forward(self, cid, feats):
        return torch.cat([self.emb(cid), self.mlp(feats)], dim=1)

In [28]:
class HybridNCF(nn.Module):
    def __init__(self, num_users, num_cards, user_feat_dim, card_feat_dim):
        super().__init__()

        self.user_encoder = UserEncoder(num_users, user_feat_dim)
        self.card_encoder = CardEncoder(num_cards, card_feat_dim)

        self.interaction = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def score(self, uid, ufeat, cid, cfeat):
        u = self.user_encoder(uid, ufeat)
        c = self.card_encoder(cid, cfeat)
        return self.interaction(torch.cat([u, c], dim=1)).squeeze(1)

    def forward(self, uid, ufeat, pos_id, pos_feat, neg_id, neg_feat):
        pos_score = self.score(uid, ufeat, pos_id, pos_feat)
        neg_score = self.score(uid, ufeat, neg_id, neg_feat)
        return pos_score, neg_score

In [29]:
def bpr_loss(pos, neg):
    return -torch.mean(F.logsigmoid(pos - neg))

In [30]:
model = HybridNCF(
    num_users=len(user_id_map),
    num_cards=len(card_id_map),
    user_feat_dim=len(user_feat_cols),
    card_feat_dim=len(card_feat_cols)
)

In [31]:
dataset = PairwiseNCFDataset(df8, df5, df7)

loader = DataLoader(
    dataset,
    batch_size=128,
    shuffle=True
)

In [32]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [35]:
model.train()

for batch in loader:
    uid, ufeat, pos_id, pos_feat, neg_id, neg_feat = batch

    pos_score, neg_score = model(
        uid, ufeat, pos_id, pos_feat, neg_id, neg_feat
    )

    loss = bpr_loss(pos_score, neg_score)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    print("Loss:", loss.item())
    break

TypeError: can't convert np.ndarray of type numpy.object_. The only supported types are: float64, float32, float16, complex64, complex128, int64, int32, int16, int8, uint64, uint32, uint16, uint8, and bool.

In [37]:
def __getitem__(self, idx):
    row = self.pairs.iloc[idx]

    user_id = int(row['user_id'])
    pos_card = row['pos_card_id']
    neg_card = row['neg_card_id']

    user_vals = self.user_df.loc[user_id, user_feat_cols].values
    pos_vals = self.card_df.loc[pos_card, card_feat_cols].values
    neg_vals = self.card_df.loc[neg_card, card_feat_cols].values

    # 🔥 DEBUG PRINTS (temporary)
    if user_vals.dtype == object:
        raise ValueError("USER FEATURES OBJECT:", user_vals, user_vals.dtype)

    if pos_vals.dtype == object:
        raise ValueError("POS CARD FEATURES OBJECT:", pos_vals, pos_vals.dtype)

    if neg_vals.dtype == object:
        raise ValueError("NEG CARD FEATURES OBJECT:", neg_vals, neg_vals.dtype)

    user_feats = torch.tensor(user_vals, dtype=torch.float32)
    pos_feats = torch.tensor(pos_vals, dtype=torch.float32)
    neg_feats = torch.tensor(neg_vals, dtype=torch.float32)

    return (
        torch.tensor(user_id_map[user_id]),
        user_feats,
        torch.tensor(card_id_map[pos_card]),
        pos_feats,
        torch.tensor(card_id_map[neg_card]),
        neg_feats
    )

In [40]:
df7['card_id'] = df7['card_id'].astype(str).str.strip()
df8['pos_card_id'] = df8['pos_card_id'].astype(str).str.strip()
df8['neg_card_id'] = df8['neg_card_id'].astype(str).str.strip()

In [41]:
card_id_map = {cid: idx for idx, cid in enumerate(df7['card_id'].unique())}

In [42]:
assert df7['card_id'].is_unique, "card_id is not unique in df7"
assert set(df8['pos_card_id']).issubset(set(df7['card_id']))
assert set(df8['neg_card_id']).issubset(set(df7['card_id']))

In [44]:
class PairwiseNCFDataset(Dataset):
    def __init__(self, pairwise_df, user_df, card_df):
        self.pairs = pairwise_df.reset_index(drop=True)
        self.user_df = user_df.set_index('User')
        self.card_df = card_df.set_index('card_id')

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        row = self.pairs.iloc[idx]

        user_id = int(row['user_id'])
        pos_card = row['pos_card_id']
        neg_card = row['neg_card_id']

        # ---- FEATURES (already numeric) ----
        user_feats = torch.tensor(
            self.user_df.loc[user_id, user_feat_cols].values,
            dtype=torch.float32
        )

        pos_feats = torch.tensor(
            self.card_df.loc[pos_card, card_feat_cols].values,
            dtype=torch.float32
        )

        neg_feats = torch.tensor(
            self.card_df.loc[neg_card, card_feat_cols].values,
            dtype=torch.float32
        )

        # ---- IMPORTANT: return PYTHON INTS for IDs ----
        return (
            int(user_id_map[user_id]),
            user_feats,
            int(card_id_map[pos_card]),
            pos_feats,
            int(card_id_map[neg_card]),
            neg_feats
        )

In [45]:
dataset = PairwiseNCFDataset(df8, df5, df7)

loader = DataLoader(
    dataset,
    batch_size=128,
    shuffle=True
)

In [46]:
model.train()

for batch in loader:
    uid, ufeat, pos_id, pos_feat, neg_id, neg_feat = batch

    pos_score, neg_score = model(
        uid, ufeat, pos_id, pos_feat, neg_id, neg_feat
    )

    loss = bpr_loss(pos_score, neg_score)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    print("Loss:", loss.item())
    break

TypeError: can't convert np.ndarray of type numpy.object_. The only supported types are: float64, float32, float16, complex64, complex128, int64, int32, int16, int8, uint64, uint32, uint16, uint8, and bool.

In [47]:
# ---- USER FEATURE MATRIX ----
user_features_matrix = (
    df5
    .sort_values('User')
    [user_feat_cols]
    .to_numpy(dtype=np.float32)
)

# ---- CARD FEATURE MATRIX ----
card_features_matrix = (
    df7
    .sort_values('card_id')
    [card_feat_cols]
    .to_numpy(dtype=np.float32)
)

In [48]:
user_id_map = {
    uid: idx for idx, uid in enumerate(df5.sort_values('User')['User'])
}

card_id_map = {
    cid: idx for idx, cid in enumerate(df7.sort_values('card_id')['card_id'])
}

In [49]:
class PairwiseNCFDataset(Dataset):
    def __init__(self, pairwise_df):
        self.pairs = pairwise_df.reset_index(drop=True)

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        row = self.pairs.iloc[idx]

        u = user_id_map[int(row['user_id'])]
        p = card_id_map[row['pos_card_id']]
        n = card_id_map[row['neg_card_id']]

        return (
            u,
            torch.from_numpy(user_features_matrix[u]),
            p,
            torch.from_numpy(card_features_matrix[p]),
            n,
            torch.from_numpy(card_features_matrix[n])
        )

In [50]:
dataset = PairwiseNCFDataset(df8)

loader = DataLoader(
    dataset,
    batch_size=128,
    shuffle=True
)

In [51]:
 model.train()

for batch in loader:
    uid, ufeat, pos_id, pos_feat, neg_id, neg_feat = batch

    pos_score, neg_score = model(
        uid, ufeat, pos_id, pos_feat, neg_id, neg_feat
    )

    loss = bpr_loss(pos_score, neg_score)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    print("Loss:", loss.item())
    break

Loss: 78.23249053955078


In [52]:
from sklearn.model_selection import train_test_split

# unique users in pairwise data
users = df8['user_id'].unique()

train_users, val_users = train_test_split(
    users, test_size=0.2, random_state=42
)

train_df = df8[df8['user_id'].isin(train_users)]
val_df   = df8[df8['user_id'].isin(val_users)]

print(len(train_df), len(val_df))

6696 1674


In [53]:
train_dataset = PairwiseNCFDataset(train_df)
val_dataset   = PairwiseNCFDataset(val_df)

train_loader = DataLoader(
    train_dataset, batch_size=256, shuffle=True
)

val_loader = DataLoader(
    val_dataset, batch_size=256, shuffle=False
)

In [54]:
def train_ncf(
    model,
    train_loader,
    val_loader,
    optimizer,
    epochs=10
):
    for epoch in range(1, epochs + 1):
        # ---- TRAIN ----
        model.train()
        train_loss = 0.0

        for batch in train_loader:
            uid, ufeat, pos_id, pos_feat, neg_id, neg_feat = batch

            pos_score, neg_score = model(
                uid, ufeat, pos_id, pos_feat, neg_id, neg_feat
            )

            loss = bpr_loss(pos_score, neg_score)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_loss += loss.item()

        train_loss /= len(train_loader)

        # ---- VALIDATION ----
        model.eval()
        val_loss = 0.0

        with torch.no_grad():
            for batch in val_loader:
                uid, ufeat, pos_id, pos_feat, neg_id, neg_feat = batch

                pos_score, neg_score = model(
                    uid, ufeat, pos_id, pos_feat, neg_id, neg_feat
                )

                loss = bpr_loss(pos_score, neg_score)
                val_loss += loss.item()

        val_loss /= len(val_loader)

        print(
            f"Epoch {epoch:02d} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Val Loss: {val_loss:.4f}"
        )

In [55]:
train_ncf(
    model,
    train_loader,
    val_loader,
    optimizer,
    epochs=10
)

Epoch 01 | Train Loss: 9.7324 | Val Loss: 2.8242
Epoch 02 | Train Loss: 1.1526 | Val Loss: 0.6031
Epoch 03 | Train Loss: 0.5517 | Val Loss: 0.4773
Epoch 04 | Train Loss: 0.6074 | Val Loss: 0.7456
Epoch 05 | Train Loss: 0.6386 | Val Loss: 0.3554
Epoch 06 | Train Loss: 0.3057 | Val Loss: 0.3228
Epoch 07 | Train Loss: 0.5308 | Val Loss: 0.3906
Epoch 08 | Train Loss: 0.3229 | Val Loss: 0.3238
Epoch 09 | Train Loss: 0.3005 | Val Loss: 0.5435
Epoch 10 | Train Loss: 0.2824 | Val Loss: 0.3700


In [56]:
def score_cards_for_user(model, user_id, candidate_card_ids):
    model.eval()
    scores = []

    u_idx = user_id_map[user_id]
    u_feat = torch.from_numpy(user_features_matrix[u_idx]).unsqueeze(0)

    with torch.no_grad():
        for cid in candidate_card_ids:
            c_idx = card_id_map[cid]
            c_feat = torch.from_numpy(card_features_matrix[c_idx]).unsqueeze(0)

            score = model.score(
                torch.tensor([u_idx]),
                u_feat,
                torch.tensor([c_idx]),
                c_feat
            )
            scores.append(score.item())

    return scores

In [57]:
import numpy as np

def dcg_at_k(relevance, k):
    relevance = np.asarray(relevance)[:k]
    return np.sum((2**relevance - 1) / np.log2(np.arange(2, len(relevance) + 2)))

def ndcg_at_k(relevance, k):
    ideal = sorted(relevance, reverse=True)
    return dcg_at_k(relevance, k) / (dcg_at_k(ideal, k) + 1e-8)

In [58]:
from collections import defaultdict

val_positives = defaultdict(set)

for _, row in val_df.iterrows():
    val_positives[row['user_id']].add(row['pos_card_id'])

In [59]:
def evaluate_ndcg(model, k=5):
    ndcgs = []

    for user_id, pos_cards in val_positives.items():
        candidate_cards = list(card_id_map.keys())

        scores = score_cards_for_user(
            model,
            user_id,
            candidate_cards
        )

        ranked_cards = [
            c for _, c in sorted(
                zip(scores, candidate_cards),
                reverse=True
            )
        ]

        relevance = [
            1 if c in pos_cards else 0
            for c in ranked_cards
        ]

        ndcgs.append(ndcg_at_k(relevance, k))

    return np.mean(ndcgs)

In [60]:
ndcg_5 = evaluate_ndcg(model, k=5)
ndcg_10 = evaluate_ndcg(model, k=10)

print("NDCG@5 :", ndcg_5)
print("NDCG@10:", ndcg_10)

NDCG@5 : 0.3715092281328118
NDCG@10: 0.5869898444688249


In [61]:
def get_eligible_cards(user_id, df5, df7):
    user_row = df5[df5['User'] == user_id].iloc[0]
    user_fico = user_row['fico_score']
    user_cluster = user_row['cluster']

    eligible = df7[
        (df7['min_fico'] <= user_fico) &
        (df7['max_fico'] >= user_fico)
    ].copy()

    # soft preference: same or adjacent cluster
    eligible['cluster_match'] = (
        eligible['cluster_id'] == user_cluster
    ).astype(int)

    return eligible

In [62]:
def score_eligible_cards(model, user_id, eligible_cards):
    model.eval()

    u_idx = user_id_map[user_id]
    u_feat = torch.from_numpy(user_features_matrix[u_idx]).unsqueeze(0)

    scores = []

    with torch.no_grad():
        for _, row in eligible_cards.iterrows():
            cid = row['card_id']
            c_idx = card_id_map[cid]
            c_feat = torch.from_numpy(card_features_matrix[c_idx]).unsqueeze(0)

            s = model.score(
                torch.tensor([u_idx]),
                u_feat,
                torch.tensor([c_idx]),
                c_feat
            ).item()

            scores.append(s)

    eligible_cards = eligible_cards.copy()
    eligible_cards['model_score'] = scores

    return eligible_cards

In [63]:
def apply_risk_penalty(cards_df, user_row):
    risk = user_row['avg_fraud_score']

    # scale penalty (tunable)
    penalty = 1 - min(risk * 5, 0.3)

    cards_df['final_score'] = cards_df['model_score'] * penalty
    return cards_df


In [64]:
def diversify(cards_df, k=5):
    selected = []
    used_brands = set()

    for _, row in cards_df.sort_values('final_score', ascending=False).iterrows():
        if row['brand'] not in used_brands:
            selected.append(row)
            used_brands.add(row['brand'])
        if len(selected) == k:
            break

    return pd.DataFrame(selected)

In [65]:
def recommend_cards_for_user(
    user_id,
    model,
    df5,
    df7,
    k=5
):
    user_row = df5[df5['User'] == user_id].iloc[0]

    eligible = get_eligible_cards(user_id, df5, df7)
    scored = score_eligible_cards(model, user_id, eligible)
    scored = apply_risk_penalty(scored, user_row)

    recommendations = diversify(scored, k=k)

    return recommendations[
        ['card_id', 'card_name', 'brand', 'card_tier',
         'annual_fee', 'apr', 'final_score', 'cluster_id']
    ]

In [66]:
recommend_cards_for_user(
    user_id=0,
    model=model,
    df5=df5,
    df7=df7,
    k=5
)

,card_id,card_name,brand,card_tier,annual_fee,apr,final_score,cluster_id
10,C3_AMEX_1,Amex Platinum Elite,Amex,premium,550,16.9,7418.508853,3


In [70]:
def get_eligible_cards(user_id, df5, df7):
    user_row = df5[df5['User'] == user_id].iloc[0]
    user_fico = user_row['fico_score']
    user_cluster = user_row['cluster']

    eligible = df7[
        (df7['min_fico'] <= user_fico) &
        (df7['max_fico'] >= user_fico)
    ].copy()

    eligible['cluster_match'] = (
        eligible['cluster_id'] == user_cluster
    ).astype(int)

    return eligible

In [71]:
def score_eligible_cards(model, user_id, eligible_cards):
    model.eval()

    u_idx = user_id_map[user_id]
    u_feat = torch.from_numpy(user_features_matrix[u_idx]).unsqueeze(0)

    scores = []

    with torch.no_grad():
        for _, row in eligible_cards.iterrows():
            cid = row['card_id']
            c_idx = card_id_map[cid]
            c_feat = torch.from_numpy(card_features_matrix[c_idx]).unsqueeze(0)

            score = model.score(
                torch.tensor([u_idx]),
                u_feat,
                torch.tensor([c_idx]),
                c_feat
            ).item()

            scores.append(score)

    eligible_cards = eligible_cards.copy()
    eligible_cards['model_score'] = scores

    return eligible_cards

In [72]:
def apply_risk_penalty(cards_df, user_row):
    risk = user_row['avg_fraud_score']
    penalty = 1 - min(risk * 5, 0.3)

    cards_df['final_score'] = cards_df['model_score'] * penalty
    return cards_df

In [73]:
def diversify_k(cards_df, k=10):
    selected_rows = []
    selected_card_ids = set()
    used_brands = set()

    sorted_df = cards_df.sort_values('final_score', ascending=False)

    # 1️⃣ Brand-diverse pass
    for _, row in sorted_df.iterrows():
        cid = row['card_id']
        brand = row['brand']

        if cid not in selected_card_ids and brand not in used_brands:
            selected_rows.append(row)
            selected_card_ids.add(cid)
            used_brands.add(brand)

        if len(selected_rows) == k:
            return pd.DataFrame(selected_rows)

    # 2️⃣ Fill remaining slots by score
    for _, row in sorted_df.iterrows():
        cid = row['card_id']

        if cid not in selected_card_ids:
            selected_rows.append(row)
            selected_card_ids.add(cid)

        if len(selected_rows) == k:
            break

    return pd.DataFrame(selected_rows)

In [74]:
def recommend_cards_for_user(user_id, model, df5, df7, k=10):
    user_row = df5[df5['User'] == user_id].iloc[0]

    eligible = get_eligible_cards(user_id, df5, df7)
    scored = score_eligible_cards(model, user_id, eligible)
    scored = apply_risk_penalty(scored, user_row)

    diversified = diversify_k(scored, k=k)

    if diversified.empty:
        return pd.DataFrame()

    diversified = diversified.reset_index(drop=True)
    diversified['user_id'] = user_id
    diversified['rank'] = diversified.index + 1
    diversified['cluster'] = user_row['cluster']

    return diversified[
        [
            'user_id', 'cluster', 'rank',
            'card_id', 'card_name', 'brand', 'card_tier',
            'annual_fee', 'apr', 'final_score', 'cluster_id'
        ]
    ]

In [75]:
all_recommendations = []

for user_id in df5['User'].unique():
    try:
        recs = recommend_cards_for_user(
            user_id=user_id,
            model=model,
            df5=df5,
            df7=df7,
            k=10
        )

        if not recs.empty:
            all_recommendations.append(recs)

    except Exception as e:
        print(f"User {user_id} failed: {e}")

In [76]:
final_recommendation_df = pd.concat(
    all_recommendations,
    ignore_index=True
)

In [81]:
cluster_persona_map = {
    0: "Retail-Centric Lifestyle Spenders",
    1: "Fuel & Commute Power Users",
    2: "Dining-Focused Light Users",
    3: "Affluent Multi-Category Spenders",
    4: "Budget-Conscious Essentials Users",
    5: "Dining & Convenience Seekers",
    6: "Digital & Lifestyle Natives"
}

final_recommendation_df['persona'] = (
    final_recommendation_df['cluster']
    .map(cluster_persona_map)
)

In [82]:
# Shape check
final_recommendation_df.shape

(11517, 12)

In [83]:
# Exactly 10 cards per user
final_recommendation_df.groupby('user_id').size().value_counts()

10    543
6     337
8     184
1     179
2     168
7     165
3     117
5     100
9       8
Name: count, dtype: int64

In [84]:
# No missing scores
final_recommendation_df['final_score'].isna().sum()

np.int64(0)

In [85]:
final_recommendation_df.to_csv(
    "final_user_card_recommendations_top10.csv",
    index=False
)